In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from cycler import cycler
from datasetsforecast.m3 import M3
import warnings

warnings.filterwarnings("ignore")

In [ ]:
plt.rcParams["font.size"] = 14
plt.rcParams["axes.labelsize"] = 14
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["xtick.labelsize"] = 14
plt.rcParams["ytick.labelsize"] = 14
plt.rcParams["legend.fontsize"] = 14
plt.rcParams["lines.linewidth"] = 2
plt.rcParams["axes.prop_cycle"] = cycler(
    color=[
        "#000072",  # blue (for historical data)
        "#80c21d",  # green (for actual data)
        "#924eae",  # purple
        "#ff0000",  # red
        "#ff9100",  # orange
    ]
)

## Load M3 dataset

In [ ]:
Y_df, *_ = M3.load(directory="../data/", group="Monthly")
Y_df["ds"] = pd.to_datetime(Y_df["ds"])

print(Y_df.head())
print(f"Number of unique time series: {Y_df['unique_id'].nunique()}")

In [ ]:
fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(14, 12))

for i, ax in enumerate(axes.flatten()):
    id = f"M{i+1}"
    filtered_df = Y_df[Y_df["unique_id"] == id]

    ax.plot(filtered_df["ds"], filtered_df["y"])
    ax.set_xlabel("Date")
    ax.set_ylabel("Value")
    ax.set_title(f"M{i+1}")

plt.tight_layout()

plt.savefig("figures/CH02_F03_peixeiro.png", dpi=300)
plt.savefig("figures/CH02_F03_peixeiro.pdf", format="pdf", bbox_inches="tight")

## Train model

In [ ]:
from neuralforecast.core import NeuralForecast
from neuralforecast.models import NBEATS

In [ ]:
# use only 1 GPU if avail
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
horizon = 12
models = [NBEATS(input_size=2 * horizon, h=horizon, max_steps=1000)]

In [ ]:
nf = NeuralForecast(models=models, freq="M")
nf.fit(df=Y_df)

### Save model

In [ ]:
nf.save(path="./model", model_index=None, overwrite=True, save_dataset=False)

### Load model

In [ ]:
pretrained_model = NeuralForecast.load(path="./model")

## Transfer learning
### Load data

In [ ]:
df = pd.read_csv("../data/AusAntidiabeticDrug.csv")
df["ds"] = pd.to_datetime(df["ds"])
df["ds"] = df["ds"] + pd.offsets.MonthEnd(0)
df.insert(0, "unique_id", 1)

df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(df["ds"], df["y"])
ax.set_xlabel("Date")
ax.set_ylabel("Volume of anti-diabetic drug prescriptions")

plt.tight_layout()

plt.savefig("figures/CH02_F04_peixeiro.png", dpi=300)
plt.savefig("figures/CH02_F04_peixeiro.pdf", format="pdf", bbox_inches="tight")

In [ ]:
input_df = df[:-12]
test_df = df[-12:]

### Zero-shot forecasting

In [ ]:
zero_shot_preds = pretrained_model.predict(input_df)
zero_shot_preds.head()

### Fine-tuning

In [ ]:
def set_max_steps(nf, max_steps):
    trainer_kwargs = {**{"max_steps": max_steps}}
    nf.models[0].trainer_kwargs = trainer_kwargs


set_max_steps(pretrained_model, 10)

In [ ]:
pretrained_model.fit(input_df)

In [ ]:
finetuned_preds = pretrained_model.predict()
finetuned_preds.head()

### Data-specific model

In [ ]:
horizon = 12

In [ ]:
models = [NBEATS(input_size=2 * horizon, h=horizon, max_steps=100)]

nf = NeuralForecast(models=models, freq="M")
nf.fit(df=input_df)

In [ ]:
trained_preds = nf.predict()
trained_preds.head()

## Evaluation

In [ ]:
zero_shot_preds = zero_shot_preds.rename(columns={"NBEATS": "NBEATS_pretrained"})
finetuned_preds = finetuned_preds.rename(columns={"NBEATS": "NBEATS_finetuned"})
trained_preds = trained_preds.rename(columns={"NBEATS": "NBEATS_trained"})

# test_df = pd.merge(test_df, zero_shot_preds, 'left', 'ds')
# test_df = pd.merge(test_df, finetuned_preds, 'left', 'ds')
# test_df = pd.merge(test_df, trained_preds, 'left', 'ds')

# Merge on both unique_id and ds to avoid duplicates
test_df = pd.merge(test_df, zero_shot_preds, on=["unique_id", "ds"], how="left")
test_df = pd.merge(test_df, finetuned_preds, on=["unique_id", "ds"], how="left")
test_df = pd.merge(test_df, trained_preds, on=["unique_id", "ds"], how="left")

test_df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(df["ds"].iloc[-60:], df["y"].iloc[-60:])
ax.plot(test_df["ds"], test_df["y"], label="Actual")
ax.plot(test_df["ds"], test_df["NBEATS_pretrained"], ls="--", label="Zero-shot")
ax.plot(test_df["ds"], test_df["NBEATS_finetuned"], ls="-.", label="Fine-tuned")
ax.plot(test_df["ds"], test_df["NBEATS_trained"], ls=":", label="Trained")

ax.set_xlabel("Date")
ax.set_ylabel("Volume of anti-diabetic drug prescriptions")

ax.legend(loc="best")

plt.tight_layout()

plt.savefig("figures/CH02_F05_peixeiro.png", dpi=300)
plt.savefig("figures/CH02_F05_peixeiro.pdf", format="pdf", bbox_inches="tight")

In [ ]:
from utilsforecast.losses import mae, smape
from utilsforecast.evaluation import evaluate

evaluation = evaluate(
    test_df,
    metrics=[mae, smape],
    models=["NBEATS_pretrained", "NBEATS_finetuned", "NBEATS_trained"],
    target_col="y",
)

evaluation = evaluation.drop(["unique_id"], axis=1)
evaluation = evaluation.set_index("metric")
evaluation

In [ ]:
ax = evaluation.plot(kind="bar", figsize=(10, 6), rot=0)

# Adding labels and title
plt.xlabel("Metric")
plt.ylabel("Value")
plt.title("Comparison of MAE and sMAPE for Different Models")
plt.legend(title="Model")

# Show plot
plt.show()

## Forecasting another frequency

In [ ]:
daily_df = pd.read_csv("../data/daily_min_temp.csv")
daily_df = daily_df.rename(columns={"Date": "ds", "Temp": "y"})
daily_df["ds"] = pd.to_datetime(daily_df["ds"])
daily_df.insert(0, "unique_id", 1)

daily_df.head()

In [ ]:
d_input_df = daily_df[:-12]
d_test_df = daily_df[-12:]

### Zero-shot forecasting

In [ ]:
pretrained_model = NeuralForecast.load(path="./model")

d_zero_shot_preds = pretrained_model.predict(d_input_df)
d_zero_shot_preds.head()

### Training a model

In [ ]:
models = [NBEATS(input_size=2 * horizon, h=horizon, max_steps=500)]

nf = NeuralForecast(models=models, freq="D")
nf.fit(df=d_input_df)

In [ ]:
d_trained_preds = nf.predict()
d_trained_preds.head()

### Evaluation

In [ ]:
d_zero_shot_preds = d_zero_shot_preds.rename(columns={"NBEATS": "NBEATS_zero_shot"})
d_zero_shot_preds = d_zero_shot_preds.reset_index(drop=True)
d_trained_preds = d_trained_preds.rename(columns={"NBEATS": "NBEATS_trained"})

# d_test_df = pd.merge(d_test_df, d_trained_preds, "left", "ds")
d_test_df = pd.merge(d_test_df, d_trained_preds, on=["unique_id", "ds"], how="left")
d_test_df = pd.concat([d_test_df, d_zero_shot_preds["NBEATS_zero_shot"]], axis=1)

print(d_test_df)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(daily_df["ds"].iloc[-60:], daily_df["y"].iloc[-60:])
ax.plot(d_test_df["ds"], d_test_df["y"], label="Actual")
ax.plot(d_test_df["ds"], d_test_df["NBEATS_zero_shot"], ls="--", label="Zero-shot")
ax.plot(d_test_df["ds"], d_test_df["NBEATS_trained"], ls=":", label="Trained")

ax.set_xlabel("Date")
ax.set_ylabel("Minimum temperature (Celsius)")

ax.legend(loc="best")

plt.tight_layout()
fig.autofmt_xdate()

plt.savefig("figures/CH02_F06_peixeiro.png", dpi=300)
plt.savefig("figures/CH02_F06_peixeiro.pdf", format="pdf", bbox_inches="tight")

In [ ]:
d_evaluation = evaluate(
    d_test_df,
    metrics=[mae, smape],
    models=["NBEATS_zero_shot", "NBEATS_trained"],
    target_col="y",
)

d_evaluation = d_evaluation.drop(["unique_id"], axis=1)
d_evaluation = d_evaluation.set_index("metric")
d_evaluation